# How many failure types are still undiscovered?

This notebook runs the pilot on Colab using every CPU core in parallel. No GPU is needed: choose **Runtime → Change runtime type → CPU** (High-RAM if available).

Run the cells in order. The pilot (3 datasets × 3 models × 5 seeds) takes about 10–20 minutes on 8 cores.

In [ ]:
# 1. Get the code and check it
!git clone -q https://github.com/ParitKansal/undiscovered-failures.git
%cd undiscovered-failures
!pip install -q -r requirements.txt
!python -m tests.test_estimators
!python -m tests.test_end_to_end | tail -1
import os; print('CPU cores:', os.cpu_count())

In [ ]:
# 2. Optional: save results to Google Drive so they survive a disconnect
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/undiscovered-failures'
os.makedirs(OUT, exist_ok=True)

In [ ]:
# 3. Run the pilot in parallel (all cores)
!python experiment.py --datasets adult,bank,covertype --models logreg,forest,boosting --seeds 5 --repeats 30 --out {OUT}/pilot.csv --data_home /content/data

In [ ]:
# 4. Summary tables
!python summarize.py {OUT}/pilot.csv

## Pilot v1: wider design and bootstrap intervals
4 datasets (adds MAGIC) × 3 models × 3 seeds × 2 flip rates (0.6, 0.9) × 3 numbers of planted types (15, 30, 60) = 216 jobs.
Each row now also has bootstrap 95% intervals for Chao1, ACE and the jackknife. About 30–40 minutes on 8 cores (faster with more).

In [ ]:
# 5. Run pilot v1 (all cores)
!git pull -q
!python -m tests.test_estimators
!python experiment.py --datasets adult,bank,covertype,magic --models logreg,forest,boosting --seeds 3 --repeats 30 --flips 0.6,0.9 --k_planted 15,30,60 --boot 200 --out {OUT}/pilot_v1.csv --data_home /content/data

In [ ]:
# 6. Summary tables for v1
!python summarize.py {OUT}/pilot_v1.csv